# Camada Gold — Modelagem Star Schema para Power BI

**Disciplina:** Mineração de Dados — FATEC Matão-SP (Prof. Rodrigo Vidrich)  
**Projeto:** Trabalho Final (P2) — Arquitetura Medallion (`Bronze → Silver → Gold → Power BI`)  
**Integrantes:** Jerônimo Barbieri Junior, Luan Chierato Oliveira, Miguel Miranda Morandini, Victor Hugo Malipense Testi  
**Data:** 13/11/2026  

### Modelo Estrela (Star Schema — Aula 04)
- **Dimensões:** `dim_aluno`, `dim_curso`, `dim_calendario`
- **Fatos:** `fato_matriculas`, `fato_notas`, `fato_financeiro`, `fato_aluno_360` (visão consolidada dos 376.925 alunos matriculados com indicadores acadêmicos, de engajamento, sociais, financeiros e de deslocamento).

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "notebooks" / "utils").exists():
        if str(candidate / "notebooks") not in sys.path:
            sys.path.insert(0, str(candidate / "notebooks"))
        break

from utils.gold_utils import (
    compute_descriptive_stats,
    get_gold_paths,
    save_gold_json,
    save_gold_table,
)

ROOT_DIR, SILVER_DIR, GOLD_DIR = get_gold_paths()
print(f"Silver: {SILVER_DIR} | Gold: {GOLD_DIR}")

Silver: C:\Projects\trabalho-final-mineacao-de-dados\data\silver | Gold: C:\Projects\trabalho-final-mineacao-de-dados\data\gold


In [2]:
alunos = pd.read_parquet(SILVER_DIR / "alunos.parquet")
matriculas = pd.read_parquet(SILVER_DIR / "matriculas.parquet")
notas = pd.read_parquet(SILVER_DIR / "notas.parquet")
acessos = pd.read_parquet(SILVER_DIR / "acessos.parquet")
redes = pd.read_parquet(SILVER_DIR / "redes.parquet")
financeiro = pd.read_parquet(SILVER_DIR / "financeiro.parquet")
deslocamento = pd.read_parquet(SILVER_DIR / "deslocamento.parquet")

# 1. dim_curso
dim_curso = (
    matriculas[["faculdade", "curso"]]
    .drop_duplicates()
    .sort_values(["faculdade", "curso"])
    .reset_index(drop=True)
)
dim_curso["id_curso"] = np.arange(1, len(dim_curso) + 1, dtype="int64")
dim_curso = dim_curso[["id_curso", "faculdade", "curso"]]
save_gold_table(dim_curso, "dim_curso", GOLD_DIR)

# 2. dim_calendario (cobre intervalo de notas e pagamentos)
datas_validas = pd.concat([notas["data"], financeiro["data_pagamento"]]).dropna()
dt_min = datas_validas.min().normalize()
dt_max = datas_validas.max().normalize()
cal_range = pd.date_range(dt_min, dt_max, freq="D")
dim_calendario = pd.DataFrame({"data": cal_range})
dim_calendario["ano"] = dim_calendario["data"].dt.year
dim_calendario["mes"] = dim_calendario["data"].dt.month
dim_calendario["ano_mes"] = dim_calendario["data"].dt.strftime("%Y-%m")
dim_calendario["trimestre"] = "T" + dim_calendario["data"].dt.quarter.astype(str)
save_gold_table(dim_calendario, "dim_calendario", GOLD_DIR)

# 3. dim_aluno (LGPD: apenas nome_anonimizado + atributos demográficos)
mat_ids = set(matriculas["id_aluno"].dropna())
nota_ids = set(notas["id_aluno"].dropna())
acesso_ids = set(acessos["id_aluno"].dropna())
fin_ids = set(financeiro["id_aluno"].dropna())

dim_aluno = alunos[[
    "id_aluno", "nome_anonimizado", "sexo", "idade", "faixa_etaria",
    "cidade", "renda_estimada", "faixa_renda"
]].copy()
dim_aluno["tem_matricula"] = dim_aluno["id_aluno"].isin(mat_ids)
dim_aluno["tem_nota"] = dim_aluno["id_aluno"].isin(nota_ids)
dim_aluno["tem_acesso"] = dim_aluno["id_aluno"].isin(acesso_ids)
dim_aluno["tem_financeiro"] = dim_aluno["id_aluno"].isin(fin_ids)
save_gold_table(dim_aluno, "dim_aluno", GOLD_DIR)

# 4. fato_matriculas
fato_matriculas = matriculas.merge(dim_curso, on=["faculdade", "curso"], how="left")
fato_matriculas["flag_evadido"] = (fato_matriculas["status"] == "Evadido").astype("int8")
fato_matriculas["flag_trancado"] = (fato_matriculas["status"] == "Trancado").astype("int8")
fato_matriculas["flag_ativo"] = (fato_matriculas["status"] == "Ativo").astype("int8")
save_gold_table(fato_matriculas, "fato_matriculas", GOLD_DIR)

# 5. fato_notas (exporta Parquet e CSV para BI)
fato_notas = notas[[
    "id_aluno", "disciplina", "data", "nota", "nota_bruta", "flag_nota_invalida", "situacao_nota"
]].copy()
save_gold_table(fato_notas, "fato_notas", GOLD_DIR)

# 6. fato_financeiro
fato_financeiro = financeiro[[
    "id_lancamento", "id_aluno", "competencia", "valor_mensalidade",
    "valor_desconto", "valor_pago", "situacao_financeira", "forma_pagamento", "data_pagamento"
]].copy()
fato_financeiro["flag_inadimplente"] = fato_financeiro["situacao_financeira"].isin(["EM_ATRASO", "PENDENTE"]).astype("int8")
save_gold_table(fato_financeiro, "fato_financeiro", GOLD_DIR)

# 7. fato_aluno_360 (agregação por aluno matriculado para Mineração e KPIs 360)
agg_mat = fato_matriculas.groupby("id_aluno", as_index=False).agg(
    total_matriculas=("id_matricula", "count"),
    qtd_ativo=("flag_ativo", "sum"),
    qtd_evadido=("flag_evadido", "sum"),
    qtd_trancado=("flag_trancado", "sum"),
    faculdade_principal=("faculdade", "first"),
    curso_principal=("curso", "first"),
    semestre_principal=("semestre", "first"),
)
agg_mat["evadido_target"] = (agg_mat["qtd_evadido"] > 0).astype("int8")

agg_notas = fato_notas.groupby("id_aluno", as_index=False).agg(
    nota_media=("nota", "mean"),
    nota_mediana=("nota", "median"),
    qtd_notas_validas=("nota", "count"),
    qtd_notas_invalidas=("flag_nota_invalida", "sum"),
)

agg_acessos = acessos.groupby("id_aluno", as_index=False).agg(
    total_acessos=("tempo_logado", "count"),
    tempo_logado_total=("tempo_logado", "sum"),
    tempo_logado_medio=("tempo_logado", "mean"),
    paginas_total=("paginas", "sum"),
)

agg_redes = redes.groupby("id_aluno", as_index=False).agg(
    interacoes_total=("interacoes", "sum"),
    seguidores_medio=("seguidores", "mean"),
    seguindo_medio=("seguindo", "mean"),
)

agg_fin = fato_financeiro.groupby("id_aluno", as_index=False).agg(
    qtd_lancamentos_fin=("id_lancamento", "count"),
    valor_mensalidade_total=("valor_mensalidade", "sum"),
    valor_pago_total=("valor_pago", "sum"),
    qtd_inadimplente=("flag_inadimplente", "sum"),
)
agg_fin["taxa_inadimplencia"] = np.where(
    agg_fin["qtd_lancamentos_fin"] > 0,
    agg_fin["qtd_inadimplente"] / agg_fin["qtd_lancamentos_fin"],
    0.0,
)

agg_desl = deslocamento.groupby("id_aluno", as_index=False).agg(
    distancia_km_media=("distancia_km", "mean"),
    tempo_deslocamento_medio=("tempo_minutos", "mean"),
    custo_diario_medio=("custo_diario", "mean"),
)

fato_360 = (
    agg_mat
    .merge(dim_aluno[["id_aluno", "sexo", "idade", "faixa_etaria", "cidade", "renda_estimada", "faixa_renda"]], on="id_aluno", how="left")
    .merge(agg_notas, on="id_aluno", how="left")
    .merge(agg_acessos, on="id_aluno", how="left")
    .merge(agg_redes, on="id_aluno", how="left")
    .merge(agg_fin, on="id_aluno", how="left")
    .merge(agg_desl, on="id_aluno", how="left")
)

# Preenche contagens ausentes com 0 (quem nunca acessou/não tem rede)
for zero_col in ["qtd_notas_validas", "qtd_notas_invalidas", "total_acessos", "tempo_logado_total", "paginas_total", "interacoes_total", "qtd_lancamentos_fin", "qtd_inadimplente"]:
    fato_360[zero_col] = fato_360[zero_col].fillna(0)

save_gold_table(fato_360, "fato_aluno_360", GOLD_DIR)

resumo_star = {
    "dim_curso": len(dim_curso),
    "dim_calendario": len(dim_calendario),
    "dim_aluno": len(dim_aluno),
    "fato_matriculas": len(fato_matriculas),
    "fato_notas": len(fato_notas),
    "fato_financeiro": len(fato_financeiro),
    "fato_aluno_360": len(fato_360),
    "taxa_evasao_alunos_pct": round(100.0 * float(fato_360["evadido_target"].mean()), 2),
}
save_gold_json(resumo_star, "resumo_star_schema", GOLD_DIR)
print(json.dumps(resumo_star, ensure_ascii=False, indent=2))

{
  "dim_curso": 16,
  "dim_calendario": 2001,
  "dim_aluno": 1000000,
  "fato_matriculas": 700000,
  "fato_notas": 600000,
  "fato_financeiro": 993804,
  "fato_aluno_360": 376925,
  "taxa_evasao_alunos_pct": 25.14
}
